In [1]:
from pyspark.sql import SparkSession
import pyspark.sql.types as T
import pyspark.sql.functions as F
from pyspark.ml.feature import VectorAssembler
from pyspark.sql.window import Window

import os
import logging
import datetime

VBox()

Starting Spark application


ID,YARN Application ID,Kind,State,Spark UI,Driver log,User,Current session?
0,application_1731491157434_0001,pyspark,idle,Link,Link,assumed-role_SSO_MKTG_STRAT_RTI_stage1540_mediaset_it,✔


FloatProgress(value=0.0, bar_style='info', description='Progress:', layout=Layout(height='25px', width='50%'),…

SparkSession available as 'spark'.


FloatProgress(value=0.0, bar_style='info', description='Progress:', layout=Layout(height='25px', width='50%'),…

In [2]:
df=spark.table("athena_rti_mktg.ec_user_clusters")
df=df.filter(df.ym == '202312') # Training 

VBox()

FloatProgress(value=0.0, bar_style='info', description='Progress:', layout=Layout(height='25px', width='50%'),…

In [3]:
# Calculate label counts and sampling values in label_weights
label_counts = df.groupBy("label_pred_desc").count()

total_count = df.count()

label_weights = label_counts.withColumn("weight", F.col("count") / total_count)

label_weights = label_weights.withColumn("sampling",F.round( F.col("weight") * 50000))
label_weights.show()

VBox()

FloatProgress(value=0.0, bar_style='info', description='Progress:', layout=Layout(height='25px', width='50%'),…

+--------------------+------+--------------------+--------+
|     label_pred_desc| count|              weight|sampling|
+--------------------+------+--------------------+--------+
|    Satira e costume|221633|  0.0326885098412277|  1634.0|
|     Birra e costine|155938|0.022999196182975305|  1150.0|
|      Psycho-reality|597977| 0.08819524641785212|  4410.0|
|     Temptation only|363043|0.053544980568276515|  2677.0|
|  Crimine e famiglia|126543| 0.01866374637729254|   933.0|
|Defilippismo ludo...|163137|0.024060972102387116|  1203.0|
| Soap early adopters|343576|0.050673805151803426|  2534.0|
|          Info lover|297642| 0.04389902878254906|  2195.0|
|Defilippismo lacr...|183288|0.027033030242693747|  1352.0|
|       Mafia fiction|156671|0.023107305885563003|  1155.0|
| Film con leggerezza|193943|0.028604529398317147|  1430.0|
|      Tendenza Amici|291670| 0.04301822231071584|  2151.0|
|Romanticismo otto...| 92334|0.013618282781354395|   681.0|
|  New Amsterdam only| 58904| 0.00868771

In [6]:
# Join the label_weights with df to get the sampling value for each label
df_with_sampling = df.join(label_weights.select("label_pred_desc", "sampling"), on="label_pred_desc", how="left")

# Define a window partitioned by label_pred_desc and ordered by some column (e.g., unique ID or row number)
window = Window.partitionBy("label_pred_desc").orderBy(F.monotonically_increasing_id())
df_with_sampling = df_with_sampling.withColumn("row_num", F.row_number().over(window))

# Filter rows where row_num is less than or equal to the sampling value for each label_pred_desc
filtered_df = df_with_sampling.filter(F.col("row_num") <= F.col("sampling"))

# Drop the helper columns if no longer needed
filtered_df = filtered_df.drop("sampling", "row_num")

# Select IDs
sampling_ids = filtered_df.select(["user_uid", "label_pred_desc"])

sampling_ids.count()

# Store sampling IDs
sampling_ids.coalesce(1).write.mode('overwrite').parquet("s3://mediaset-mktg/pf_1540/sampling_idsandlabs")

VBox()

FloatProgress(value=0.0, bar_style='info', description='Progress:', layout=Layout(height='25px', width='50%'),…

In [5]:
sampling_ids.show()

VBox()

FloatProgress(value=0.0, bar_style='info', description='Progress:', layout=Layout(height='25px', width='50%'),…

+--------------------+----------------+
|            user_uid| label_pred_desc|
+--------------------+----------------+
|67be0f45781f4016b...|Satira e costume|
|67bee81923ba4f09b...|Satira e costume|
|67bf554bf0194b45a...|Satira e costume|
|67bf5c47e17e4cc5a...|Satira e costume|
|67bff083c43945cc8...|Satira e costume|
|67c024a1949b45579...|Satira e costume|
|67c0bbda7aec44e78...|Satira e costume|
|67c0cbc9f74543b3a...|Satira e costume|
|67c15750b0114b259...|Satira e costume|
|67c3008feab04dc49...|Satira e costume|
|67c369d6bc3146a88...|Satira e costume|
|67c37f17387b484ca...|Satira e costume|
|67c38603c53543a68...|Satira e costume|
|67c39db216394154b...|Satira e costume|
|67c3c1d1e16a4af2b...|Satira e costume|
|67c422cf246746938...|Satira e costume|
|67c4257909264a088...|Satira e costume|
|67c4a29c911c480da...|Satira e costume|
|67c4c58da5f444b19...|Satira e costume|
|67c51d4b4e0343aca...|Satira e costume|
+--------------------+----------------+
only showing top 20 rows